# Exercise 3: Pro-drop counting

*Syntax and Universal Dependencies · version 0.1.0*

> Read the Turkish IMST training set (gold annotation). Select eligible predicates: finite verbs (`VerbForm=Fin` or no VerbForm with Person/Number agreement) that are not imperatives and not passive. A missing `nsubj` is not proof of pro-drop: also count `nsubj:pass`, `csubj` and other subject relations, and treat nonfinite clauses separately. For eligible predicates, compute the proportion with an overt subject per tense-mood combination (e.g., `Tense=Past|Mood=Ind`). Manually adjudicate a random sample of 50 "subjectless" cases and report how many are genuine pro-drop versus annotation errors or other constructions. Then parse the same text with the TurkicNLP Turkish pipeline: does the parser reproduce the gold proportions?

**Setup.** `pip install "turkicnlp==0.5.0" "stanza==1.11.0" scipy matplotlib requests`. Stanza is pinned to 1.11.0 because the worked solutions use its Turkish (IMST), Kazakh (KTB) and Kyrgyz (KTMU) checkpoints; with another version, Stanza downloads that version's models and the numbers can change. Models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`); missing ones are downloaded. The Universal Dependencies files are the **UD 2.17** release files (tag `r2.17` on GitHub). They are downloaded once into `UD217_DIR` (default `~/.cache/ud217`) and checked against the SHA-256 hashes recorded in the book's treebank inventory. Parsers receive the **gold words** (`tokenize_pretokenized=True`), so tokenisation errors do not enter the scores.

**Definitions.** Eligible predicates are VERB words that are finite (`VerbForm=Fin`, or no VerbForm but Person or Number; IMST leaves VerbForm empty on most finite verbs), not imperative and not passive. Participles, converbs and verbal nouns are therefore excluded. A predicate has an overt subject if it has a dependent whose relation is `nsubj` or `csubj` or one of their subtypes. One construction needs special care: in IMST a light verb depends on its noun (*yok oldu*: *oldu* → `compound:lvc` of *yok*), and the subject is attached to the noun, so for light verbs the noun's dependents are checked as well. The parser comparison uses the IMST **test** set, because the Stanza model was trained on the training split.

In [ ]:
import collections, hashlib, math, os, random, statistics
from pathlib import Path
import requests

# Pinned UD 2.17 release files (the inventory in tables/tbl-ud217-inventory) and their SHA-256 hashes.
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {
    "UD_Turkish-BOUN/tr_boun-ud-train.conllu": "373fe39e3acdba3219521f54fa5ae768851fbb4bf9a31079ad1ca6590410650c",
    "UD_Turkish-BOUN/tr_boun-ud-test.conllu": "4f74fc0f9efdd1285d781f3efd1e98cbf39cf4433df9eb4dc14162e8ed36773b",
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Turkish-TueCL/tr_tuecl-ud-test.conllu": "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    "UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu": "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    "UD_Kyrgyz-TueCL/ky_tuecl-ud-test.conllu": "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
    "UD_English-EWT/en_ewt-ud-train.conllu": "6fe5692f2e02198262483daba0626efe346dd19f0ce6c3ee7728b08af9b90a9c",
}
CACHE = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()

def ud_path(key):
    """Local copy of a pinned UD 2.17 file, downloaded once and checked against its SHA-256 hash."""
    path = CACHE / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos xpos feats head deprel")

def read(key_or_path):
    """Sentences as dicts: sent_id, text and words (integer-ID syntactic words only;
    multiword-token ranges and empty nodes are skipped)."""
    path = ud_path(key_or_path) if key_or_path in SHA256 else Path(key_or_path)
    sents, meta, words = [], {}, []
    for line in path.read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append({"sent_id": meta.get("sent_id"), "text": meta.get("text"), "words": words})
            meta, words = {}, []
        elif line.startswith("#"):
            k, _, v = line[1:].partition("=")
            meta.setdefault(k.strip(), v.strip())
        else:
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[4], c[5], int(c[6]), c[7]))
    return sents

def feats(w):
    return dict(f.split("=", 1) for f in w.feats.split("|") if "=" in f)

def children(words):
    out = collections.defaultdict(list)
    for w in words:
        out[w.head].append(w)
    return out

def subtree(words, i, kids=None):
    """IDs of word i and all its descendants."""
    kids = kids or children(words)
    out, stack = set(), [i]
    while stack:
        j = stack.pop(); out.add(j); stack.extend(k.id for k in kids[j])
    return out

def q(values, p):
    """Nearest-rank quantile, as in the chapter's arc-length table."""
    v = sorted(values)
    return v[math.ceil(len(v) * p) - 1] if v else None

def stanza_pipeline(lang, processors="tokenize,pos,lemma,depparse"):
    """A Stanza pipeline over gold words (tokenize_pretokenized=True). Uses models already on disk
    (STANZA_RESOURCES_DIR or ~/stanza_resources) and downloads them only if they are missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "depparse").exists()
    return stanza.Pipeline(lang, processors=processors, tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def parse_gold_words(nlp, sents, forms=None):
    """Parse each gold sentence from its gold words (optionally replaced by `forms`, one list per
    sentence, e.g. transliterated) and return predicted sentences with the original word forms."""
    forms = forms or [[w.form for w in s["words"]] for s in sents]
    doc = nlp(forms)
    out = []
    for s, ps in zip(sents, doc.sentences):
        assert len(ps.words) == len(s["words"]), s["sent_id"]
        out.append({"sent_id": s["sent_id"], "text": s["text"], "words": [
            Word(g.id, g.form, p.lemma or "_", p.upos or "_", p.xpos or "_", p.feats or "_", p.head, p.deprel)
            for g, p in zip(s["words"], ps.words)]})
    return out

def write_conllu(sents, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in sents:
            f.write(f"# sent_id = {s['sent_id']}\n# text = {s['text']}\n")
            for w in s["words"]:
                f.write("\t".join([str(w.id), w.form, w.lemma or "_", w.upos, w.xpos or "_", w.feats or "_",
                                   str(w.head), w.deprel, "_", "_"]) + "\n")
            f.write("\n")

def score(gold, pred):
    """UPOS, UAS and LAS (%) with the official CoNLL 2018 scorer, over gold-word input.
    Gold multiword tokens are not written, so both files have the same words and text."""
    import tempfile
    from stanza.utils import conll18_ud_eval as ev
    with tempfile.TemporaryDirectory() as tmp:
        g, p = Path(tmp) / "gold.conllu", Path(tmp) / "pred.conllu"
        write_conllu(gold, g); write_conllu(pred, p)
        r = ev.evaluate(ev.load_conllu_file(str(g)), ev.load_conllu_file(str(p)))
    return {k: round(100 * r[k].f1, 2) for k in ("Words", "UPOS", "UAS", "LAS")}

In [ ]:
SUBJECT_RELS = ("nsubj", "csubj")      # with subtypes such as nsubj:pass and csubj:pass

def eligible(w):
    """Finite, non-imperative, non-passive verbal predicates. IMST leaves VerbForm empty on most
    finite verbs, so 'finite' is VerbForm=Fin or no VerbForm with Person or Number agreement."""
    f = feats(w)
    finite = f.get("VerbForm") == "Fin" or ("VerbForm" not in f and ("Person" in f or "Number" in f))
    return (w.upos == "VERB" and finite and f.get("Mood") != "Imp"
            and "Pass" not in f.get("Voice", ""))

def predicates(sents):
    """(sentence, predicate, has overt subject) for every eligible predicate."""
    out = []
    for s in sents:
        kids = children(s["words"])
        for w in s["words"]:
            if eligible(w):
                # Under the IMST/BOUN convention a light verb depends on its noun (compound:lvc),
                # and the subject is attached to the noun, so look there as well.
                heads = [w.id] + ([w.head] if w.deprel.startswith("compound") else [])
                overt = any(k.deprel.split(":")[0] in SUBJECT_RELS for h in heads for k in kids[h])
                out.append((s, w, overt))
    return out

def table(rows, key, min_n=30):
    groups = collections.defaultdict(list)
    for s, w, overt in rows:
        groups[key(w)].append(overt)
    for k, v in sorted(groups.items(), key=lambda x: -len(x[1])):
        if len(v) >= min_n:
            print(f"   {k:28} n={len(v):5}  overt subject {sum(v) / len(v):.1%}")

imst_train = read("UD_Turkish-IMST/tr_imst-ud-train.conllu")
rows = predicates(imst_train)
print(f"IMST train: {len(rows)} eligible predicates, overt subject {sum(o for *_, o in rows) / len(rows):.1%}")
print(" by Tense|Mood (n >= 30):")
table(rows, lambda w: f"Tense={feats(w).get('Tense', '-')}|Mood={feats(w).get('Mood', '-')}")
print(" by Person:")
table(rows, lambda w: f"Person={feats(w).get('Person', '-')}", min_n=1)
lv = [o for s, w, o in rows if w.deprel.startswith("compound")]
print(f" light verbs (deprel compound*) among eligible predicates: {len(lv)}, of which {sum(lv)} have the subject on the noun")

# A reproducible sample of 50 subjectless predicates, with a first automatic triage.
def triage(s, w):
    f = feats(w); kids = children(s["words"]); by_id = {x.id: x for x in s["words"]}
    if f.get("Person") in ("1", "2"):
        return "1st/2nd person: pro-drop"
    if w.deprel == "conj" and any(k.deprel.split(":")[0] == "nsubj" for k in kids[w.head]):
        return "3rd person conjunct: subject of first conjunct"
    return "3rd person: check by hand"
random.seed(8)
sample = random.sample([(s, w) for s, w, o in rows if not o], 50)
cats = collections.Counter(triage(s, w) for s, w in sample)
print("sample of 50 subjectless predicates:", dict(cats))
for s, w in sample:
    print(f"   {s['sent_id']:14} {w.form:16} {feats(w).get('Person', '-')}  {triage(s, w):45} {s['text'][:60]}")

In [ ]:
# Does the parser reproduce the gold proportion? Stanza (IMST models) on the IMST test set, from gold words.
imst_test = read("UD_Turkish-IMST/tr_imst-ud-test.conllu")
nlp_tr = stanza_pipeline("tr")
imst_pred = parse_gold_words(nlp_tr, imst_test)
print("Stanza on IMST test vs gold:", score(imst_test, imst_pred))
for label, sents in [("gold", imst_test), ("Stanza", imst_pred)]:
    r = predicates(sents)
    by_p = collections.defaultdict(list)
    for s, w, o in r:
        by_p[feats(w).get("Person", "-")].append(o)
    print(f"IMST test, {label:6}: {len(r)} eligible predicates, overt subject {sum(o for *_, o in r) / len(r):.1%}; "
          + ", ".join(f"Person={p} {sum(v) / len(v):.1%} (n={len(v)})" for p, v in sorted(by_p.items())))

## Worked solution

Output of `experiments/ch08_exercises/run_ch08_exercises.py` (UD 2.17; Stanza 1.11.0):

```
IMST train: 3223 eligible predicates, overt subject 43.8%
 by Tense|Mood (n >= 30):
   Tense=Past|Mood=Ind          n= 1574  overt subject 45.2%
   Tense=Pres|Mood=Ind          n=  882  overt subject 44.9%
   Tense=Pqp|Mood=Ind           n=  136  overt subject 44.1%
   Tense=Fut|Mood=Ind           n=  129  overt subject 46.5%
   Tense=Pres|Mood=Pot          n=  118  overt subject 41.5%
   Tense=Pres|Mood=Opt          n=   89  overt subject 9.0%
   Tense=Pres|Mood=Des          n=   65  overt subject 32.3%
   Tense=Pres|Mood=Cnd          n=   51  overt subject 27.5%
   Tense=Past|Mood=Pot          n=   51  overt subject 33.3%
   Tense=Past|Mood=Gen          n=   47  overt subject 74.5%
 by Person:
   Person=3                     n= 2243  overt subject 56.9%
   Person=1                     n=  808  overt subject 13.5%
   Person=2                     n=  172  overt subject 15.1%
 light verbs (deprel compound*) among eligible predicates: 527, of which 225 have the subject on the noun
sample of 50 subjectless predicates: {'3rd person: check by hand': 20, '1st/2nd person: pro-drop': 28, '3rd person conjunct: subject of first conjunct': 2}
   00053223_66    sordu            3  3rd person: check by hand                     İyi, ama neden? diye sordu sonunda.
   00095233_101   beceremiyordum   1  1st/2nd person: pro-drop                      İntihar etmek istiyor ama beceremiyordum.
   00095233_121   yakalarız        1  1st/2nd person: pro-drop                      Yaşamın ruhunu duygularla yakalarız.
   00032161_192   içiyorduk        1  1st/2nd person: pro-drop                      O günün son içkilerini içiyorduk.
   00053223_4     başladık         1  1st/2nd person: pro-drop                      Bir o, bir ben, hiç önceden kararlaştırmadan, kendiliğinden 
   00172176_75    boyattı          3  3rd person: check by hand                     Saçlarını boyattı, modelini değiştirdi.
   00006231_141   taşıyor          3  3rd person conjunct: subject of first conjunct Öyle bir felsefe ki bu, kendi içinde bir çelişkiyi taşıyor; 
   00032161_84    geldiler         3  3rd person: check by hand                     Öğretmenim, diyeceğim, köpeklerden korktuğum için Miskoye il
   00032161_204   istiyorum        1  1st/2nd person: pro-drop                      İşte geldik gidiyoruz, dedim, sana bir şey sormak istiyorum.
   00058111_8     kullanacağım     1  1st/2nd person: pro-drop                      Öldürmekle de kalmayıp cesedinizi kadavra niyetine kullanaca
   20270000_46    uyudum           1  1st/2nd person: pro-drop                      Uyandım ve sabah rolü bana verdiklerine dair telefon alacağı
   00111211_135   belirttiler      3  3rd person: check by hand                     Bunun üzerine askeri savcılıktan bir defa daha talepte bulun
   00053223_32    yıkar            3  3rd person conjunct: subject of first conjunct Ölüm ancak çatısı kapanan evlere girer, dağıtır, parçalar, y
   00105133_44    bekledim         1  1st/2nd person: pro-drop                      Oturup sigara yaktım, telaşe halimin geçmesini bekledim.
   00166271_14    dedi             3  3rd person: check by hand                     Eee birini seviyorsan onunla beraber yaşamanın bedelini öder
   00006231_105   duyarsam         1  1st/2nd person: pro-drop                      Sıcak, hazır, güzel yiyecekler; gereksinme duyarsam soğuk be
   00105133_189   öptü             3  3rd person: check by hand                     Baktı olmayacak, beni yanağımdan öptü, saçlarımı okşadı.
   00111211_35    dedi             3  3rd person: check by hand                     Bu yola başvurun, dedi.
   00105133_181   oldu             3  3rd person: check by hand                     Saçmalama, neler reddedildi, nelerden vazgeçildi, neler iyiy
   00105133_10    bilsen           2  1st/2nd person: pro-drop                      Halbuki arada gidip günah çıkardın mı, kendini bir miktar yı
   00111211_84    edeceksiniz      2  1st/2nd person: pro-drop                      Şimdi birini oradan oraya alarak tayin edeceksiniz, öbürünü 
   00131266_326   gittik           1  1st/2nd person: pro-drop                      Meclis'e gittik.
   00053223_2     Özlemiş          3  3rd person: check by hand                     Özlemiş beni.
   20580000_55    demiş            3  3rd person: check by hand                     Deşifre metnini dinleyen Arınç, " Bunu demiş olabilirim, ama
   00105133_46    gittim           1  1st/2nd person: pro-drop                      Baktım baktım yürüdüm gittim.
   00032161_108   aldı             3  3rd person: check by hand                     Başına gelecek her şeye hazır bir duruş aldı.
   00111211_28    Sanıyorum        1  1st/2nd person: pro-drop                      Sanıyorum, zaman aşımına uğradığı için benim itiraz sürem de
   00053223_71    sordum           1  1st/2nd person: pro-drop                      Ara ara aklıma takılan soruları da anlatır gibi, ona mı, ken
   00172176_270   oturduk          1  1st/2nd person: pro-drop                      Önlerde bir yere oturduk.
   00006231_67    bulmuşsun        2  1st/2nd person: pro-drop                      Ne rahat, ne güzel bir ev bulmuşsun.
   00172176_62    anlatamam        1  1st/2nd person: pro-drop                      Halamın çocukları da o kadar yaramaz ki anlatamam.
   00058111_55    girdin           2  1st/2nd person: pro-drop                      Bir akşam üstü de başındaki o komik perukla fahişe olmadan ö
   00131266_12    hatırlamıyorum   1  1st/2nd person: pro-drop                      Tarihini hatırlamıyorum, ağabeyim, Firuz bu Babayasa yı görü
   00105133_58    bekliyordu       3  3rd person: check by hand                     Beni bekliyordu.
   00105133_217   önerseydin       2  1st/2nd person: pro-drop                      Israr etseydin bize, daha güzel bir isim önerseydin!..
   00095233_135   öğreniyorlar     3  3rd person: check by hand                     NA'nın on iki basamağında, tıpkı bir bebek gibi duygularıyla
   00172176_143   geldi            3  3rd person: check by hand                     Bak şimdi birdenbire aklıma geldi.
   00032161_160   duracaksın       2  1st/2nd person: pro-drop                      Aaa, diyorlar, kimi elini saygıyla göğsüne bastırıyor kimi d
   00166271_76    Yattık           1  1st/2nd person: pro-drop                      Siz yatmadınız mı? diye sordum Özer'e Yattık ama sonra sizin
   00058111_30    kuramazsınız     2  1st/2nd person: pro-drop                      Ama bir türlü, Ben de yıllar önce sizin gibi diye başlayan b
   00032161_136   olurmuş          3  3rd person: check by hand                     Hem bir şey kırk kere söylenirse olurmuş.
   20270000_52    yapacağız        1  1st/2nd person: pro-drop                      Ama verdikleri çekleri, ' Nakit ödeme yapacağız ' diye aldıl
   00032161_16    dedi             3  3rd person: check by hand                     Hiiç, dedi, oturuyorum.
   00105133_4     işledin          2  1st/2nd person: pro-drop                      Bizde, günahı işledin mi bitti.
   00142211_244   buluyor          3  3rd person: check by hand                     Patlıcan, zeytinyağlı, sağyağlı, salata, turşu derken gerçek
   20580000_54    isterseniz       2  1st/2nd person: pro-drop                      Ankara Temsilcisi Fikret Bila da, Arınç'ı arayarak, " Bir se
   00095233_129   yöneliyor        3  3rd person: check by hand                     Yetmiyor, yine favori maddesine yöneliyor.
   00032161_154   diyor            3  3rd person: check by hand                     Ne kadar zengin olursan ol, diyor, seyirci olduktan sonra (a
   00166271_69    sanırsın         2  1st/2nd person: pro-drop                      Beni görür görmez kendini yere atıp göbeğini bir açışı var k
   00016112_64    ulaşmakta        3  3rd person: check by hand                     Bu atılımın son dalgaları, sonuna ulaşmakta olduğumuz yüzyıl
```

```
Stanza on IMST test vs gold: {'Words': 100.0, 'UPOS': 95.18, 'UAS': 78.37, 'LAS': 70.23}
IMST test, gold  : 1066 eligible predicates, overt subject 36.2%; Person=1 11.9% (n=327), Person=2 12.6% (n=87), Person=3 51.5% (n=652)
IMST test, Stanza: 1082 eligible predicates, overt subject 34.4%; Person=1 17.4% (n=328), Person=2 17.0% (n=88), Person=3 45.0% (n=666)
```

**Proportions.** 3,223 predicates in the IMST training split are eligible, and 43.8% of them have an overt subject. Person matters far more than tense or mood: 56.9% of third-person predicates have an overt subject, but only 13.5% of first-person and 15.1% of second-person ones, whose agreement suffix identifies the referent. Among the frequent tense–mood combinations the indicative tenses lie between 44% and 47%. The optative (9.0%) is low because 60 of its 89 eligible forms are first person, and the conditional (27.5%) has a first- or second-person majority as well (30 of 55 forms). The generalising `Mood=Gen` (*-DIr*) is high (74.5%); all 75 of its forms are third person. Without the light-verb check the overall figure would be 36.9%: 225 of the 527 light verbs have their subject on the noun, and counting them as subjectless would overstate pro-drop.

**The sample.** The 50 subjectless predicates printed above (random seed 8) are triaged automatically: 28 are first or second person, which in Turkish is pro-drop almost by definition, and 2 are third-person conjuncts whose first conjunct has the subject (shared subject in coordination, not discourse pro-drop). The remaining 20 third-person cases need a reader. For example, *… diye sordu* '… (s)he asked' is ordinary third-person pro-drop with an antecedent in the preceding discourse. The remaining third-person cases are left for the reader to adjudicate.

**The parser.** On the IMST test set the gold trees give 36.2% predicates with an overt subject, and Stanza's trees give 34.4%. The total is close, but the errors do not cancel by person: the parser attaches a subject to too many first- and second-person predicates (17.4% and 17.0% against 11.9% and 12.6%) and to too few third-person ones (45.0% against 51.5%). Tagging and parsing errors also change the set of eligible predicates (1,082 against 1,066). A pro-drop statistic computed from parser output should therefore be reported with the gold comparison.

## Your turn — extension

1. Extend the eligibility to nominal predicates with a copula or person suffix (*öğrenciyim*, *evdeydi*). How does the overt-subject rate change?
2. Repeat the gold count on BOUN. Do the two treebanks agree once the light-verb convention is handled?